# Model: protectai/deberta-v3-base-prompt-injection-v2

## 🚀 RMIT Hackathon 2026 — Official Model Training Template
> **Notebook Naming:** `<university>_<campus>_<team>_C1` (VD: `RMIT_SGS_TeamName_C1`)
> **BTC Strict Rule:** You must build your solution on ONE approved model (no multi-backbone ensembling). State the chosen model at the very top line.

### 10 Approved Models Whitelist:
1. `protectai/deberta-v3-base-prompt-injection-v2` (Khuyên dùng - Bảo mật an toàn)
2. `patronus-studio/wolf-defender-prompt-injection` (Bảo mật an toàn)
3. `FacebookAI/xlm-roberta-base` (Rất mạnh về Romance: pt, fr, it)
4. `FacebookAI/xlm-roberta-large` (Cực mạnh nhưng nặng; dùng batch size 4)
5. `jhu-clsp/mmBERT-base` (Đa ngôn ngữ hiện đại, nhẹ, train nhanh)
6. `jhu-clsp/mmBERT-small` (Siêu nhẹ)
7. `distilbert/distilbert-base-multilingual-cased` (Nhanh nhất)
8. `google/rembert` (32 layers đa ngôn ngữ; batch size 8)
9. `ibm-granite/granite-3.3-2b-instruct` (LLM)
10. `mistralai/Mistral-7B-Instruct-v0.3` (LLM)

In [ ]:
# =============================================================================
# ⚙️ 1. CẤU HÌNH SIÊU THAM SỐ (CHỈ CẦN THAY ĐỔI TẠI CELL NÀY)
# =============================================================================
# Chọn 1 trong 10 model approved ở trên
MODEL_NAME = "protectai/deberta-v3-base-prompt-injection-v2"

# Cấu hình huấn luyện
BATCH_SIZE = 8          # Nếu model là xlm-roberta-large thì giảm xuống 4
GRAD_ACCUM_STEPS = 2    # Effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS = 16
EPOCHS = 3              # 3-4 epochs là tối ưu
LEARNING_RATE = 2e-5    # 1.5e-5 cho model large, 2e-5 cho model base
MAX_LENGTH = 512        # Cắt Head 256 + Tail 256 tokens
SEED = 42
TARGET_FOLD = 0         # Huấn luyện trên Fold 0 (hoặc đổi fold khác)

print(f"[*] Đã chọn Model: {MODEL_NAME}")
print(f"[*] Effective Batch Size: {BATCH_SIZE * GRAD_ACCUM_STEPS} | Epochs: {EPOCHS} | LR: {LEARNING_RATE}")

In [ ]:
# =============================================================================
# 📦 2. CÀI ĐẶT & IMPORT THƯ VIỆN
# =============================================================================
import os
import re
import gc
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Thiết bị tính toán: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"[*] GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
# 🧹 3. HÀM LÀM SẠCH VĂN BẢN AN TOÀN (BẢO TOÀN DẤU ROMANCE & KÝ TỰ BẺ KHÓA)
# =============================================================================
def clean_text_safety(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Chuẩn hóa Unicode NFC (giữ nguyên dấu é, è, ê, ç, ã, õ, á, í...)
    text = unicodedata.normalize("NFC", text)
    # Xóa ký tự điều khiển tàng hình nhưng giữ lại xuống dòng
    text = "".join(ch for ch in text if ch in ("\n", "\r", "\t") or unicodedata.category(ch)[0] != "C")
    # Gộp khoảng trắng thừa
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Bản đồ nhãn chuẩn
LABEL2ID = {"BENIGN": 0, "JAILBREAK": 1, "PROMPT_INJECTION": 2}
ID2LABEL = {0: "BENIGN", 1: "JAILBREAK", 2: "PROMPT_INJECTION"}

In [ ]:
# =============================================================================
# 📂 4. ĐỌC DỮ LIỆU & ÁP DỤNG PREFIX NGÔN NGỮ
# =============================================================================
def find_file(filename):
    candidates = [
        Path("/kaggle/input") / filename,
        Path(f"/kaggle/input/rmit-hackathon-2026/{filename}"),
        Path(f"data/raw/{filename}"),
        Path(f"../data/raw/{filename}"),
        Path(f"data/processed/{filename}"),
        Path(f"../data/processed/{filename}")
    ]
    for p in candidates:
        if p.exists():
            return p
    if Path("/kaggle/input").exists():
        found = list(Path("/kaggle/input").rglob(filename))
        if found:
            return found[0]
    raise FileNotFoundError(f"Không tìm thấy {filename}")

train_path = find_file("train.csv")
test_path = find_file("test.csv")
print(f"[*] Train: {train_path} | Test: {test_path}")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df["clean_text"] = train_df["text"].apply(clean_text_safety)
train_df["formatted_text"] = "lang: " + train_df["language"].fillna("en") + " | " + train_df["clean_text"]
train_df["label_id"] = train_df["label"].map(LABEL2ID)

test_df["clean_text"] = test_df["text"].apply(clean_text_safety)
test_df["formatted_text"] = "lang: " + test_df["language"].fillna("en") + " | " + test_df["clean_text"]

try:
    folds_path = find_file("folds.csv")
    folds_df = pd.read_csv(folds_path)
    train_df["fold"] = folds_df["fold"]
    print("[*] Đã nạp 5-Fold thống nhất từ folds.csv")
except Exception:
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_df["fold"] = -1
    for f, (_, val_idx) in enumerate(skf.split(train_df, train_df["label_id"])):
        train_df.loc[val_idx, "fold"] = f
    print("[*] Đã sinh Stratified 5-Fold mới")

print(f"[*] Số dòng: Train = {len(train_df)}, Test = {len(test_df)}")

In [ ]:
# =============================================================================
# ✂️ 5. TOKENIZER VỚI HEAD + TAIL TRUNCATION (CẮT ĐẦU 256 + ĐUÔI 256)
# =============================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token

class SecurityDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.half_len = (max_len - 2) // 2

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = self.tokenizer.encode(text, add_special_tokens=False)

        # Head + Tail Truncation
        if len(tokens) > self.max_len - 2:
            tokens = tokens[:self.half_len] + tokens[-self.half_len:]

        cls_id = self.tokenizer.cls_token_id or self.tokenizer.bos_token_id or 0
        sep_id = self.tokenizer.sep_token_id or self.tokenizer.eos_token_id or 0
        input_ids = [cls_id] + tokens + [sep_id]
        attention_mask = [1] * len(input_ids)

        pad_len = self.max_len - len(input_ids)
        if pad_len > 0:
            pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0
            input_ids += [pad_id] * pad_len
            attention_mask += [0] * pad_len

        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long)
        }
        if self.labels is not None:
            item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# =============================================================================
# 🏋️ 6. THIẾT LẬP DATALOADER & CLASS-WEIGHTED LOSS
# =============================================================================
train_split = train_df[train_df["fold"] != TARGET_FOLD].reset_index(drop=True)
val_split = train_df[train_df["fold"] == TARGET_FOLD].reset_index(drop=True)

train_ds = SecurityDataset(train_split["formatted_text"], train_split["label_id"].values, tokenizer, MAX_LENGTH)
val_ds = SecurityDataset(val_split["formatted_text"], val_split["label_id"].values, tokenizer, MAX_LENGTH)
test_ds = SecurityDataset(test_df["formatted_text"], None, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

# Trọng số phạt: Lớp 7% PROMPT_INJECTION phạt x8.8 lần
class_weights = torch.tensor([1.0, 2.0, 8.8], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(f"[*] Đã nạp Class-Weighted Loss: {class_weights.tolist()}")

In [ ]:
# =============================================================================
# 🚀 7. KHỞI TẠO MÔ HÌNH & HUẤN LUYỆN (TRAINING LOOP)
# =============================================================================
# Quan trọng: ignore_mismatched_sizes=True giúp các model pre-train sẵn 2 nhãn
# (như protectai/deberta-v3-base-prompt-injection-v2, wolf-defender) tự động
# khởi tạo lại lớp classifier 3 nhãn mà không bị lỗi RuntimeError lệch shape.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True
)
if getattr(model.config, "pad_token_id", None) is None and tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda") if DEVICE.type == "cuda" else None

best_macro_f1 = -1.0
best_val_probs = None
best_checkpoint_path = "best_model.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        if scaler:
            with torch.amp.autocast("cuda"):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
        else:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels) / GRAD_ACCUM_STEPS
            loss.backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()

        train_loss += loss.item() * GRAD_ACCUM_STEPS
        pbar.set_postfix({"loss": f"{train_loss / (step + 1):.4f}"})

    # --- ĐÁNH GIÁ TRÊN TẬP VALIDATION ---
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            if scaler:
                with torch.amp.autocast("cuda"):
                    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            else:
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
            val_preds.append(probs)
            val_targets.append(batch["label"].numpy())

    val_probs = np.nan_to_num(np.concatenate(val_preds, axis=0), nan=1.0 / 3.0)
    val_targets = np.concatenate(val_targets, axis=0)
    val_pred_labels = np.argmax(val_probs, axis=1)
    macro_f1 = f1_score(val_targets, val_pred_labels, average="macro")
    per_cls_f1 = f1_score(val_targets, val_pred_labels, average=None, labels=[0, 1, 2], zero_division=0)
    print(
        f"\n[Epoch {epoch}] Val Macro-F1 = {macro_f1:.4f} | "
        f"BENIGN: {per_cls_f1[0]:.4f} | JAILBREAK: {per_cls_f1[1]:.4f} | PROMPT_INJECTION: {per_cls_f1[2]:.4f}"
    )

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_val_probs = val_probs.copy()
        torch.save(model.state_dict(), best_checkpoint_path)
        print(f"[*] Checkpoint mới tốt nhất đã được lưu: {best_macro_f1:.4f}")

print(f"\n[*] ĐÃ XONG TRAIN! Best Raw Validation Macro-F1: {best_macro_f1:.4f}")

In [ ]:
# =============================================================================
# 🔮 8. TỐI ƯU NGƯỠNG XÁC SUẤT, INFERENCE TẬP TEST & LƯU FILE (.CSV + .NPY)
# =============================================================================
def optimize_class_multipliers(val_probs, val_targets):
    """Tìm bộ hệ số nhân xác suất [m0, m1, m2] tối ưu hóa trực tiếp Macro-F1."""
    best_mult = np.array([1.0, 1.0, 1.0], dtype=float)
    best_score = f1_score(val_targets, np.argmax(val_probs, axis=1), average="macro")
    grid = np.linspace(0.4, 2.2, 37)
    for _ in range(2):
        for c in [1, 2]:
            for m in grid:
                cand = best_mult.copy()
                cand[c] = m
                preds = np.argmax(val_probs * cand, axis=1)
                score = f1_score(val_targets, preds, average="macro")
                if score > best_score + 1e-5:
                    best_score = score
                    best_mult = cand
    return best_mult, best_score

best_multipliers, tuned_macro_f1 = optimize_class_multipliers(best_val_probs, val_targets)
print(f"[*] Multipliers tối ưu trên Val: {np.round(best_multipliers, 3).tolist()}")
print(f"[*] Validation Macro-F1 sau khi tối ưu ngưỡng: {tuned_macro_f1:.4f} (Gốc: {best_macro_f1:.4f})")
best_macro_f1 = max(best_macro_f1, tuned_macro_f1)

model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE))
model.eval()

test_preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting Test"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        if scaler:
            with torch.amp.autocast("cuda"):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        else:
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        test_preds.append(torch.softmax(logits.float(), dim=1).cpu().numpy())

test_probs = np.nan_to_num(np.concatenate(test_preds, axis=0), nan=1.0 / 3.0)
test_pred_ids = np.argmax(test_probs * best_multipliers, axis=1)

sub_df = pd.DataFrame({
    "id": test_df["id"],
    "label": [ID2LABEL[idx] for idx in test_pred_ids]
})

out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("submissions")
out_dir.mkdir(parents=True, exist_ok=True)

model_slug = MODEL_NAME.split("/")[-1].replace("-", "_")
sub_path = out_dir / "submission.csv"
sub_named_path = out_dir / f"submission_{model_slug}_fold{TARGET_FOLD}.csv"
val_probs_path = out_dir / f"val_probs_{model_slug}_fold{TARGET_FOLD}.npy"
test_probs_path = out_dir / f"test_probs_{model_slug}_fold{TARGET_FOLD}.npy"

sub_df.to_csv(sub_path, index=False)
sub_df.to_csv(sub_named_path, index=False)
np.save(val_probs_path, best_val_probs)
np.save(test_probs_path, test_probs)

print(f"[*] Đã lưu file nộp chính tại: {sub_path}")
print(f"[*] Đã lưu bản sao theo tên model: {sub_named_path}")
print(f"[*] Đã lưu xác suất để Ensemble/Blend sáng mai: {test_probs_path}")

In [ ]:
# =============================================================================
# 🔍 9. KIỂM TRA ĐẠT CHUẨN TRƯỚC KHI BẤM NỘP (SANITY CHECK)
# =============================================================================
print("="*55)
print("🔍 KẾT QUẢ KIỂM TRA FILE NỘP (READY TO SUBMIT CHECK)")
print("="*55)

assert len(sub_df) == 6000, f"LỖI: Số dòng {len(sub_df)} khác 6000!"
assert list(sub_df.columns) == ["id", "label"], f"LỖI: Cột {sub_df.columns} sai!"
assert sub_df.isna().sum().sum() == 0, "LỖI: Có giá trị rỗng/NaN!"
print("✅ 1. Kích thước (6000, 2) & Dữ liệu không NaN: HOÀN HẢO")

valid_labels = {"BENIGN", "JAILBREAK", "PROMPT_INJECTION"}
actual_labels = set(sub_df["label"].unique())
assert actual_labels.issubset(valid_labels), f"LỖI: Có nhãn lạ {actual_labels}!"
print(f"✅ 2. Các nhãn dự đoán hợp lệ: {actual_labels}")

dist = sub_df["label"].value_counts(normalize=True) * 100
print("\n📊 Phân bố phần trăm nhãn trên tập Test:")
for lbl, pct in dist.items():
    print(f"   - {lbl}: {pct:.2f}%")

pi_pct = dist.get("PROMPT_INJECTION", 0.0)
if pi_pct < 4.0 or pi_pct > 12.0:
    print(f"\n⚠️ CẢNH BÁO: PROMPT_INJECTION chiếm {pi_pct:.2f}% (Bình thường là 5% - 10%). Cần xem xét trước khi nộp!")
else:
    print(f"\n🎯 PHÂN BỐ CỰC ĐẸP! PROMPT_INJECTION chiếm {pi_pct:.2f}%")

if best_macro_f1 >= 0.9332:
    print(f"🚀 KẾT LUẬN: ĐỦ ĐIỀU KIỆN NỘP KAGGLE! (Val F1: {best_macro_f1:.4f} >= Baseline 0.9332)")
else:
    print(f"❌ KẾT LUẬN: KHÔNG NÊN NỘP (Val F1: {best_macro_f1:.4f} < Baseline 0.9332)")